# Single-chain model validation

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.figures import plot_precision_recall, plot_metric_series

SWEEP = "53c_single_mol_duplex"
DENSITY = None
RG_REFERENCE_NM = 1.47

replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
if DENSITY is not None:
    replicates = filter_rows(replicates, density=DENSITY)
    summary = filter_rows(summary, density=DENSITY)

E_VALUES = sorted(summary["E_att"].dropna().unique())

## MFE-matching and additional base pairs

In [ ]:
for metric in ("mfe_matches", "extra_pairs"):
    fig, axes = panels_by(
        summary, by="angle_k_ppp",
        plot_fn=partial(
            plot_metric_series, metric=metric, style=STYLE,
            x_order=E_VALUES, band="std",
        ),
        style=STYLE, ncols=3,
    )
    fig.suptitle(STYLE.label(metric))
    plt.show()

## Precision and recall

In [ ]:
vmin, vmax = replicates["E_att"].min(), replicates["E_att"].max()
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(
        plot_precision_recall, style=STYLE, vmin=vmin, vmax=vmax,
    ),
    style=STYLE, ncols=3,
)
visible_axes = [ax for ax in axes.flat if ax.get_visible()]
fig.colorbar(visible_axes[0].collections[0], ax=visible_axes,
             label=STYLE.label("E_att"))
plt.show()

## Radius of gyration

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=partial(
        plot_metric_series, metric="Rg_nm", style=STYLE,
        x_order=E_VALUES, band="p10_p90",
    ),
    style=STYLE, ncols=3,
)
for ax in axes.flat:
    if ax.get_visible():
        ax.axhline(RG_REFERENCE_NM, linestyle="--", linewidth=1,
                   color="#666", label="Reference")
visible_axes[-1].legend(frameon=False)
plt.show()